# EDA — Arequipa subset

Exploratory only — for figuring things out interactively. Anything that needs to run repeatably (on a schedule, in CI, or reused later) lives in `ml/data_prep/clean_arequipa.py` instead.

Goal for this pass: confirm the filter column for the Arequipa subset and get a first read on data quality (nulls/completeness) scoped to that subset — not the full Peru dataset.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/pe_properties.csv", low_memory=False)
print("rows:", len(df))
print("columns:", list(df.columns))
df.head()

rows: 124449
columns: ['id', 'ad_type', 'start_date', 'end_date', 'created_on', 'lat', 'lon', 'l1', 'l2', 'l3', 'l4', 'l5', 'l6', 'rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered', 'price', 'currency', 'price_period', 'title', 'description', 'property_type', 'operation_type']


,id,ad_type,start_date,end_date,created_on,lat,lon,l1,l2,l3,...,bathrooms,surface_total,surface_covered,price,currency,price_period,title,description,property_type,operation_type
0,OHnwRj8WCByyWFlzr8CG2w==,Propiedad,2019-06-25,2019-07-23,2019-06-25,-12.094636,-77.003267,Perú,Lima,Lima,...,NaN,207.0,120.0,380000.0,USD,Mensual,SE VENDE CASITA SAN BORJA NORTE 446 4 HABITACI...,Se vende CASITA en condominio La casita empiez...,Casa,Venta
1,+r81Zjmgyd2Y+cguWfoHhw==,Propiedad,2019-06-25,2019-12-25,2019-06-25,-12.094851,-77.017706,Perú,Lima,Lima,...,NaN,347.0,319.0,800000.0,USD,Mensual,VENDO ELEGANTE CASA EN SAN ISIDRO,VENDO ELEGANTE CASA EN SAN ISIDRO. AMPLIA SALA...,Casa,Venta
2,TP2j5KyjkY1glXvrCBoqkA==,Propiedad,2019-06-25,2019-09-03,2019-06-25,-12.071644,-77.037922,Perú,Lima,Lima,...,NaN,500.0,500.0,2250000.0,USD,Mensual,¡ Se vende Terreno en Excelente Zona de Jesus ...,"Casa como terreno en venta, ubicada en toda un...",Casa,Venta
3,PF2/BiUsiKTU8i7mFBGq3A==,Propiedad,2019-06-25,2019-09-03,2019-06-25,-11.960901,-77.062730,Perú,Lima,Lima,...,NaN,224.0,126.0,152000.0,USD,Mensual,SE VENDE LINDA CASA EN LA MISMA AV METROPOLITA...,"LINDA CASA EN VENTA, FRENTE AL CENTRO DE IDIOM...",Casa,Venta
4,LgOzukt3/c7bYO7SNzFbUw==,Propiedad,2019-06-25,2019-07-10,2019-06-25,-12.102991,-76.938083,Perú,Lima,Lima,...,NaN,378.0,320.0,490000.0,USD,Mensual,SE VENDE HERMOSA Y AMPLIA CASA EN LA MOLINA,Se vende hermosa casa en La Molina 5 Habitacio...,Casa,Venta


## Which `l*` column is the Arequipa filter?

The dataset encodes location as a hierarchy `l1`..`l6` (country -> ... -> neighborhood) with no header names beyond that. Check cardinality/nulls of each to figure out which level is department, province, district.

In [2]:
loc_cols = ["l1", "l2", "l3", "l4", "l5", "l6"]
for col in loc_cols:
    print(f"{col:>3} -> nunique: {df[col].nunique(dropna=True):>4}  nulls: {df[col].isna().sum()}")

 l1 -> nunique:    1  nulls: 0
 l2 -> nunique:   25  nulls: 0
 l3 -> nunique:   86  nulls: 4953
 l4 -> nunique:  156  nulls: 25691
 l5 -> nunique:   20  nulls: 124394
 l6 -> nunique:    0  nulls: 124449


**Conclusion:** `l1` is constant (country=Peru, useless as filter). `l2` has 25 unique values with zero nulls — matches Peru's 25 departments exactly, and includes `"Arequipa"`. That's the department-level filter to use, and it lines up with the expected ballpark of ~12,400 Arequipa listings in this dataset. `l3` looks like province, `l4` like district (will matter later as the `distrito` feature).

In [3]:
aqp = df[df["l2"] == "Arequipa"].copy()
print("Arequipa subset rows:", len(aqp))
print()
print("l3 (province) within Arequipa department:")
print(aqp["l3"].value_counts(dropna=False))
print()
print("l4 (district) within Arequipa department, top 20:")
print(aqp["l4"].value_counts(dropna=False).head(20))

Arequipa subset rows: 12164

l3 (province) within Arequipa department:
l3
Arequipa    11524
Islay         287
Camaná        182
NaN           124
Caylloma       36
Caraveli       11
Name: count, dtype: int64

l4 (district) within Arequipa department, top 20:
l4
Arequipa                         2132
Cerro Colorado                   1963
Cayma                            1902
NaN                              1337
Yanahuara                        1174
Jose Luis Bustamante Y Rivero     969
Sachaca                           657
Paucarpata                        434
Miraflores                        357
Socabaya                          290
Alto Selva Alegre                 243
Jacobo Hunter                     152
Mariano Melgar                    114
Sabandia                           96
Tiabaya                            79
Characato                          71
Chiguata                           68
Uchumayo                           50
Yura                               35
Yarabamba       

## Completeness check, scoped to the Arequipa subset

Checking nulls on the fields that matter for the model (price, surface, district, lat/long, property/operation type) using only the 12,164 Arequipa rows.

In [4]:
key_cols = ["price", "currency", "surface_total", "surface_covered", "l4", "lat", "lon", "property_type", "operation_type"]
null_report = aqp[key_cols].isna().sum().to_frame("nulls")
null_report["pct"] = (null_report["nulls"] / len(aqp) * 100).round(1)
null_report

,nulls,pct
price,262,2.2
currency,295,2.4
surface_total,4630,38.1
surface_covered,6934,57.0
l4,1337,11.0
lat,457,3.8
lon,457,3.8
property_type,0,0.0
operation_type,0,0.0


## Categorical fields relevant to cleaning

`currency` and `price_period` matter for the currency-normalization step (need to know what units/periods actually show up before picking a fixed exchange rate and deciding whether rentals and sales prices are comparable). `operation_type` and `property_type` matter for the model features later.

In [5]:
print("currency:")
print(aqp["currency"].value_counts(dropna=False))
print()
print("price_period:")
print(aqp["price_period"].value_counts(dropna=False))
print()
print("operation_type:")
print(aqp["operation_type"].value_counts(dropna=False))
print()
print("property_type:")
print(aqp["property_type"].value_counts(dropna=False))

currency:
currency
USD    9034
PEN    2835
NaN     295
Name: count, dtype: int64

price_period:
price_period
NaN        6101
Mensual    6063
Name: count, dtype: int64

operation_type:
operation_type
Venta       8762
Alquiler    3402
Name: count, dtype: int64

property_type:
property_type
Departamento       4283
Casa               2944
Otro               2250
Lote               1709
Local comercial     641
Oficina             294
Depósito             42
Casa de campo         1
Name: count, dtype: int64


## Decision: how to define "superficie" before dropping incomplete rows

Rows without a price, a surface, or a district aren't usable for training and can't be safely imputed without inventing data, so they'll get dropped. But `surface_total` alone is null in 38.1% of the Arequipa subset — requiring it strictly would throw away over a third of the data. `surface_covered` is null even more often (57.0%), but the two fields don't overlap perfectly: some rows have one and not the other.

Two questions decide this: how much data is actually rescuable by combining both columns, and how predictive is surface for price in the first place (if it barely matters, losing rows over it isn't worth it either way).

In [6]:
n = len(aqp)
both_null = aqp["surface_total"].isna() & aqp["surface_covered"].isna()
rescued = aqp["surface_total"].isna() & aqp["surface_covered"].notna()

print(f"surface_total present: {aqp['surface_total'].notna().sum()} ({aqp['surface_total'].notna().mean()*100:.1f}%)")
print(f"both null (unusable either way): {both_null.sum()} ({both_null.mean()*100:.1f}%)")
print(f"rescued by coalescing (total null, covered present): {rescued.sum()} ({rescued.mean()*100:.1f}%)")
print()

both_present = aqp.dropna(subset=["surface_total", "surface_covered"])
print(f"rows with both present: {len(both_present)}")
print(f"covered > total (should not happen if covered <= lot size): {(both_present['surface_covered'] > both_present['surface_total']).sum()} rows")
print("total - covered, distribution:")
print((both_present["surface_total"] - both_present["surface_covered"]).describe())

surface_total present: 7534 (61.9%)
both null (unusable either way): 3431 (28.2%)
rescued by coalescing (total null, covered present): 1199 (9.9%)

rows with both present: 4031
covered > total (should not happen if covered <= lot size): 1171 rows
total - covered, distribution:
count      4031.000000
mean       -108.434135
std        4348.096879
min     -169104.000000
25%         -10.500000
50%           0.000000
75%           0.000000
max      129864.000000
dtype: float64


In [7]:
import numpy as np

# Predictive value of surface: log-log correlation with price (real estate
# prices scale roughly log-linearly with area, so this is a more honest read
# than raw linear correlation).
for col in ["surface_total", "surface_covered"]:
    sub = aqp.dropna(subset=["price", col])
    sub = sub[(sub["price"] > 0) & (sub[col] > 0)]
    corr = np.log(sub["price"]).corr(np.log(sub[col]))
    print(f"{col}: rows={len(sub)}, corr(log price, log {col})={corr:.3f}")
print()

# Final row counts under each strategy, after also requiring price, district
# (l4), and currency to be present — the other three fields that get dropped
# when missing. This is the number that actually matters for training data
# size, not the isolated surface null rate.
base_mask = aqp["price"].notna() & aqp["l4"].notna() & aqp["currency"].notna()
coalesced_surface = aqp["surface_total"].fillna(aqp["surface_covered"])
strict_total = aqp["surface_total"]

print("rows passing price + district + currency not-null:", base_mask.sum())
print("  + coalesced surface (total, fallback covered):", (base_mask & coalesced_surface.notna()).sum())
print("  + strict surface_total only:", (base_mask & strict_total.notna()).sum())

surface_total: rows=7404, corr(log price, log surface_total)=0.386
surface_covered: rows=5135, corr(log price, log surface_covered)=0.430

rows passing price + district + currency not-null: 10576
  + coalesced surface (total, fallback covered): 7508
  + strict surface_total only: 6670


**Decision: coalesce `surface_total` with `surface_covered` as fallback, then drop rows where both are null.**

Reasoning:
- Surface is only moderately predictive of price (log-log correlation ~0.39–0.43, both fields similar), so district and property type will likely carry more of the model's signal — this is not a case where losing surface data is catastrophic, but it's also not free to throw away rows over it.
- When both fields are present, their median difference is 0 — `surface_covered` is a legitimate stand-in for `surface_total` when the latter is missing, not a different quantity.
- Coalescing is a strict superset of requiring `surface_total` alone: every row kept by the strict rule is kept here too, plus ~800–1,200 more (depending on which other filters are applied first). There's no downside to taking it.
- Even the strict/worst case leaves thousands of rows — comfortably enough for a baseline regression model with default hyperparameters (no heavy tuning needed at this stage). The choice isn't about avoiding an unusably small dataset; it's about not discarding usable data for no reason.

This becomes the surface-handling rule in `ml/data_prep/clean_arequipa.py`.

## Later decisions (made while building `ml/data_prep/clean_arequipa.py`)

Why a separate script at all, instead of just cleaning the data right here in the notebook? Two reasons:

- **This cleaning logic needs to run more than once, unattended.** It has to run identically on this raw CSV today, and later on a completely different batch — a small, manually-collected set of real, current Arequipa listings used to check the 2020-trained model against today's market. A notebook run by hand doesn't guarantee that; a plain importable Python module does.
- **Duplicated logic drifts.** If the cleaning rules were written twice — once loosely in a notebook cell, once "for real" in a script — they'd eventually disagree in some small way nobody notices until it causes a bug.

So `clean_arequipa.py` holds every cleaning rule as a small, independently callable function (`deduplicate`, `coalesce_surface`, `normalize_currency`, ...), and the sections below **import and call those exact functions** rather than re-deriving the logic inline. That makes this notebook a live, executable proof of what the script actually does — if the script changes, re-running this notebook shows it — instead of a separate approximation that can quietly go stale.

In [8]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "ml" / "data_prep"))
from clean_arequipa import (
    CONTENT_COLS,
    PEN_PER_USD,
    coalesce_surface,
    deduplicate,
    drop_incomplete,
    filter_price_outliers,
    impute_geo,
    normalize_currency,
)

### Deduplication

`id` is already unique per row (checked below), so the interesting question is whether the same listing gets reposted under a different `id`/date — that's the kind of duplicate that actually matters for training data.

In [9]:
print("unique id:", aqp["id"].nunique(), "/", len(aqp))
print("fully identical rows (all columns):", aqp.duplicated(keep=False).sum())

content_dup_excess = aqp.duplicated(subset=CONTENT_COLS, keep="first").sum()
print("duplicate by content (same everything except id/dates), excess rows:", content_dup_excess)

unique id: 12164 / 12164
fully identical rows (all columns): 0
duplicate by content (same everything except id/dates), excess rows: 743


Sanity check before trusting this: a content match could be a false positive if it's just a bunch of rows with generic/null titles matching by coincidence. Checking that, plus looking at one real example.

In [10]:
dup_mask = aqp.duplicated(subset=CONTENT_COLS, keep=False)
dups = aqp[dup_mask].sort_values(CONTENT_COLS)
print("rows involved in content-duplication:", len(dups))
print("of those, title is null:", dups["title"].isna().sum(), "-> not a false-positive-from-nulls situation")
print()

group_id = dups.groupby(CONTENT_COLS, dropna=False).ngroup()
example = dups[group_id == group_id.iloc[0]]
example[["id", "start_date", "end_date", "created_on", "title", "price", "surface_total", "l4"]]

rows involved in content-duplication: 1426
of those, title is null: 0 -> not a false-positive-from-nulls situation



,id,start_date,end_date,created_on,title,price,surface_total,l4
87153,y90yBhndkaFnPJJ5der7qQ==,2019-04-03,2019-06-09,2019-04-03,"VENDO 10 HECTÁREAS O 100,000 M2, EN IRRIGACION...",35000.0,NaN,NaN
109982,iGnDK7zvVe6a9iPcFf3brg==,2019-08-31,2019-09-02,2019-08-31,"VENDO 10 HECTÁREAS O 100,000 M2, EN IRRIGACION...",35000.0,NaN,NaN


**Decision: drop reposts, keeping the earliest occurrence by `created_on`.** Implemented as `deduplicate()` in `clean_arequipa.py`.

In [11]:
deduped = deduplicate(aqp)
print(f"{len(aqp)} -> {len(deduped)} rows ({len(aqp) - len(deduped)} reposts dropped)")

12164 -> 11421 rows (743 reposts dropped)


### Currency normalization

`currency` is USD or PEN (see the value_counts in "Categorical fields" above). Needs a fixed exchange rate to convert to one unit, plus a decision on what to do with the small number of null-currency rows (can't convert an unknown currency).

**2020 PEN/USD exchange rate:** BCRP's published annual average for 2020 was **S/3.494 per USD** (monthly range: S/3.327 in January to S/3.608 in November — the sol depreciated through the pandemic). `clean_arequipa.py` uses `PEN_PER_USD = 3.5` — a clean constant essentially equal to the real average, reasonable as a fixed rate for a single year like 2020. Source: [BCRP official statistics](https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/PN01246PM/html).

Target currency: **USD**, chosen because it's already the majority in the data and the convention for real estate pricing in Peru — converting the minority (PEN) is less total conversion.

In [12]:
complete = drop_incomplete(coalesce_surface(deduped))
print("currency nulls in cleaned subset:", complete["currency"].isna().sum(), "/", len(complete))

normalized = normalize_currency(complete)
print(f"{len(complete)} -> {len(normalized)} rows ({len(complete) - len(normalized)} dropped, unknown currency)")
print(f"PEN_PER_USD = {PEN_PER_USD}")

currency nulls in cleaned subset: 14 / 6975
6975 -> 6961 rows (14 dropped, unknown currency)
PEN_PER_USD = 3.5


### Outlier filtering — why per `operation_type`, not global

Extreme prices (top/bottom 1%) should be dropped as outliers. Question: compute that percentile globally, or separately for `Venta` and `Alquiler`? Sale and rental prices are on completely different scales, so this isn't obvious.

In [13]:
print(normalized.groupby("operation_type")["price_usd"].describe(percentiles=[0.01, 0.5, 0.99]))
print()
print("global 1%/99%:", normalized["price_usd"].quantile([0.01, 0.99]).to_dict())
print("Venta 1%/99%:", normalized.loc[normalized["operation_type"] == "Venta", "price_usd"].quantile([0.01, 0.99]).to_dict())
print("Alquiler 1%/99%:", normalized.loc[normalized["operation_type"] == "Alquiler", "price_usd"].quantile([0.01, 0.99]).to_dict())

                 count           mean            std          min  \
operation_type                                                      
Alquiler        2073.0    1574.302460    5009.288869    85.714286   
Venta           4888.0  307873.568214  948315.609344  4857.142857   

                          1%       50%         99%         max  
operation_type                                                  
Alquiler          171.428571     600.0    22850.24     89000.0  
Venta           21945.920000  141500.0  3267618.50  43000000.0  

global 1%/99%: {0.01: 242.85714285714283, 0.99: 2732221.5999999843}
Venta 1%/99%: {0.01: 21945.920000000002, 0.99: 3267618.500000002}
Alquiler 1%/99%: {0.01: 171.42857142857142, 0.99: 22850.24000000004}


**Decision: compute the 1%/99% cutoff separately per `operation_type`.** The global 1st percentile ($243) is far below `Venta`'s real 1st percentile (~$21,945) — a global cut would barely touch `Venta`'s tail while wrongly discarding cheap-but-legitimate `Alquiler` listings. Implemented as `filter_price_outliers()`.

In [14]:
no_outliers = filter_price_outliers(normalized)
print(f"{len(normalized)} -> {len(no_outliers)} rows ({len(normalized) - len(no_outliers)} dropped)")

6961 -> 6825 rows (136 dropped)


### Geo imputation — turned out to be a no-op

Missing lat/long could be imputed with the district centroid, or the feature could be dropped if the subset already comes clean on it. Checking which applies after all the filtering above.

In [15]:
missing_before = int((no_outliers["lat"].isna() | no_outliers["lon"].isna()).sum())
geo_complete = impute_geo(no_outliers)
missing_after = int((geo_complete["lat"].isna() | geo_complete["lon"].isna()).sum())
print(f"rows missing lat/lon: {missing_before} before impute_geo, {missing_after} after")

rows missing lat/lon: 0 before impute_geo, 0 after


**Turned out to be a no-op:** 0 rows reach this step still missing lat/lon. The raw ~3.8% null rate fully overlapped with rows already dropped by `drop_incomplete`/`normalize_currency`/`filter_price_outliers`. `impute_geo()` is kept as a real transform anyway (not just an assertion) since it's reused later for a manually-collected batch of current real listings — a different data source that may not have that same overlap.

### Final `listings` table

Saved via `save_listings()` as `data/processed/listings.parquet`.

In [16]:
print("full pipeline:", len(aqp), "->", len(geo_complete), "rows")
print(geo_complete.shape)
geo_complete.head()

full pipeline: 12164 -> 6825 rows
(6825, 27)


,id,ad_type,start_date,end_date,created_on,lat,lon,l1,l2,l3,...,surface_covered,price,currency,price_period,title,description,property_type,operation_type,surface,price_usd
28,4oBAQqoLR7EEEN85uot6+g==,Propiedad,2019-06-25,2019-09-03,2019-06-25,-16.403866,-71.529860,Perú,Arequipa,Arequipa,...,369.0,390000.0,USD,Mensual,AMPLIO TERRENO EN VENTA <br>A una cuadra de la...,"Terreno en Venta de 368.50 m2 , ubicado en zon...",Lote,Venta,369.0,390000.0
36,/qokRUJoJ38XlM/hyFNeVA==,Propiedad,2019-06-25,2019-08-13,2019-06-25,-16.458269,-71.519351,Perú,Arequipa,Arequipa,...,121.0,65000.0,USD,Mensual,VENDO CASA EN LA CAMPIÑA SOCABAYA,CÓMODA CASA EN LA CAMPIÑA SOCABAYA IDEAL PARA ...,Casa,Venta,100.0,65000.0
82,/t1xCb4GD5AD5Z3tmE3WxQ==,Propiedad,2019-06-25,2019-10-23,2019-06-25,-16.382877,-71.536196,Perú,Arequipa,Arequipa,...,115.0,137000.0,USD,Mensual,LINDO Y AMPLIO DEPARTAMENTO EN QUINTA PRIVADA ...,El Departamento consta de: Amplia y bonita sal...,Departamento,Venta,108.0,137000.0
83,D+8ZBhTw/eFGlxO82RqO6w==,Propiedad,2019-06-25,2019-10-11,2019-06-25,-16.424814,-71.533324,Perú,Arequipa,Arequipa,...,85.0,85000.0,USD,Mensual,SE VENDE DEPARTAMENTO CON EXCELENTE UBICACIÓN ...,"SE VENDE DEPARTAMENTO CON EXCELENTE UBICACIÓN,...",Departamento,Venta,85.0,85000.0
84,BH/tF7KGYKxrNM7VJEZfbw==,Propiedad,2019-06-25,2019-09-03,2019-06-25,-16.372400,-71.552922,Perú,Arequipa,Arequipa,...,83.0,88000.0,USD,Mensual,"VENDO DEPARTAMENTO EXCELENTE UBICACIÓN 88,000 ...","VENDO DEPARTAMENTO 5to PISO CON ASCENSOR, AREA...",Departamento,Venta,83.0,88000.0
